In [1]:
import random, shutil
from pathlib import Path

random.seed(42)

seg_root  = Path(r"D:\Big_Data\Pothole_Segmentation_YOLOv8")
data_root = Path(r"D:\Big_Data\2020YOLO\final_cnn_dataset_v2")

train_pothole_dir  = data_root / "train" / "pothole"
yolov8_holdout_dir = data_root / "external_yolov8_positive_holdout" / "pothole"
yolov8_holdout_dir.mkdir(parents=True, exist_ok=True)

image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

positive_paths = []
for split in ["train", "valid"]:
    img_dir = seg_root / split / "images"
    lbl_dir = seg_root / split / "labels"
    if not img_dir.exists(): continue
    for img in img_dir.rglob("*"):
        if not img.is_file() or img.suffix.lower() not in image_exts: continue
        lbl = lbl_dir / f"{img.stem}.txt"
        if lbl.exists() and lbl.read_text(encoding="utf-8").strip():
            positive_paths.append(img)

print("YOLOv8 positive candidates:", len(positive_paths))

for p in train_pothole_dir.iterdir():
    if p.is_file() and p.name.startswith("yolov8_pos_"):
        p.unlink()
for p in yolov8_holdout_dir.iterdir():
    if p.is_file() and p.name.startswith("yolov8_pos_"):
        p.unlink()

random.shuffle(positive_paths)
n_holdout = int(round(len(positive_paths) * 0.30))
holdout  = positive_paths[:n_holdout]
to_train = positive_paths[n_holdout:]

for src in to_train:
    shutil.copy2(src, train_pothole_dir  / f"yolov8_pos_{src.name}")
for src in holdout:
    shutil.copy2(src, yolov8_holdout_dir / f"yolov8_pos_{src.name}")

print("added to train/pothole   :", len(to_train))
print("in yolov8 positive holdout:", len(holdout))

def cnt(folder): return len([p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in image_exts])
print("train pothole     =", cnt(data_root / "train" / "pothole"))
print("train non_pothole =", cnt(data_root / "train" / "non_pothole"))

YOLOv8 positive candidates: 780
added to train/pothole   : 546
in yolov8 positive holdout: 234
train pothole     = 4604
train non_pothole = 12705


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(8),
    transforms.ColorJitter(brightness=0.12, contrast=0.12),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset              = datasets.ImageFolder(data_root / "train", transform=train_transform)
val_dataset                = datasets.ImageFolder(data_root / "val",   transform=eval_transform)
test_dataset               = datasets.ImageFolder(data_root / "test",  transform=eval_transform)
challenge_dataset          = datasets.ImageFolder(data_root / "challenge_alligator_exclusive", transform=eval_transform)
external_sovit_neg_dataset = datasets.ImageFolder(data_root / "external_sovit_negative",       transform=eval_transform)
yolov8_holdout_dataset     = datasets.ImageFolder(data_root / "external_yolov8_positive_holdout", transform=eval_transform)

bs = 32
train_loader              = DataLoader(train_dataset,              batch_size=bs, shuffle=True,  num_workers=0, pin_memory=True)
val_loader                = DataLoader(val_dataset,                batch_size=bs, shuffle=False, num_workers=0, pin_memory=True)
test_loader               = DataLoader(test_dataset,               batch_size=bs, shuffle=False, num_workers=0, pin_memory=True)
challenge_loader          = DataLoader(challenge_dataset,          batch_size=bs, shuffle=False, num_workers=0, pin_memory=True)
external_sovit_neg_loader = DataLoader(external_sovit_neg_dataset, batch_size=bs, shuffle=False, num_workers=0, pin_memory=True)
yolov8_holdout_loader     = DataLoader(yolov8_holdout_dataset,     batch_size=bs, shuffle=False, num_workers=0, pin_memory=True)

pothole_idx = train_dataset.class_to_idx["pothole"]
print("class_to_idx:", train_dataset.class_to_idx)
print("train size:", len(train_dataset),
      "| yolov8 holdout size:", len(yolov8_holdout_dataset))

class_to_idx: {'non_pothole': 0, 'pothole': 1}
train size: 17309 | yolov8 holdout size: 234


In [3]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import models
from torch.cuda.amp import autocast, GradScaler
from sklearn.metrics import precision_score, recall_score, f1_score

torch.backends.cudnn.benchmark = True

bs_train = 64
bs_eval  = 128
nw       = 4

train_loader = DataLoader(train_dataset, batch_size=bs_train, shuffle=True,
                          num_workers=nw, pin_memory=True,
                          persistent_workers=True, prefetch_factor=4)
val_loader   = DataLoader(val_dataset, batch_size=bs_eval, shuffle=False,
                          num_workers=nw, pin_memory=True,
                          persistent_workers=True, prefetch_factor=4)
test_loader  = DataLoader(test_dataset, batch_size=bs_eval, shuffle=False,
                          num_workers=nw, pin_memory=True,
                          persistent_workers=True, prefetch_factor=4)
challenge_loader = DataLoader(challenge_dataset, batch_size=bs_eval, shuffle=False,
                              num_workers=nw, pin_memory=True,
                              persistent_workers=True, prefetch_factor=4)
external_sovit_neg_loader = DataLoader(external_sovit_neg_dataset, batch_size=bs_eval, shuffle=False,
                                       num_workers=nw, pin_memory=True,
                                       persistent_workers=True, prefetch_factor=4)
yolov8_holdout_loader = DataLoader(yolov8_holdout_dataset, batch_size=bs_eval, shuffle=False,
                                   num_workers=nw, pin_memory=True,
                                   persistent_workers=True, prefetch_factor=4)

def train_one_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()
    running_loss, total, correct = 0.0, 0, 0
    for images, labels in loader:
        images = images.to(device, non_blocking=True, memory_format=torch.channels_last)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast(dtype=torch.float16):
            outputs = model(images)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total

@torch.no_grad()
def collect_probs_and_labels(model, loader, device, pothole_idx):
    model.eval()
    ps, ls = [], []
    for images, labels in loader:
        images = images.to(device, non_blocking=True, memory_format=torch.channels_last)
        with autocast(dtype=torch.float16):
            out = model(images)
        probs = torch.softmax(out.float(), dim=1)[:, pothole_idx]
        ps.extend(probs.cpu().numpy().tolist())
        ls.extend(labels.numpy().tolist())
    return np.array(ps), np.array(ls)

@torch.no_grad()
def collect_probs_only(model, loader, device, pothole_idx):
    model.eval()
    ps = []
    for images, _ in loader:
        images = images.to(device, non_blocking=True, memory_format=torch.channels_last)
        with autocast(dtype=torch.float16):
            out = model(images)
        probs = torch.softmax(out.float(), dim=1)[:, pothole_idx]
        ps.extend(probs.cpu().numpy().tolist())
    return np.array(ps)

def metrics_from_threshold(y_true, y_prob, threshold, pothole_idx):
    y_pred = (y_prob >= threshold).astype(int)
    return (precision_score(y_true, y_pred, pos_label=pothole_idx),
            recall_score   (y_true, y_pred, pos_label=pothole_idx),
            f1_score       (y_true, y_pred, pos_label=pothole_idx),
            y_pred)

model_v3 = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model_v3.fc = nn.Linear(model_v3.fc.in_features, 2)
model_v3 = model_v3.to(device).to(memory_format=torch.channels_last)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_v3.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)
scaler    = GradScaler()

num_epochs = 10
selection_threshold = 0.50
save_path_v3 = "best_resnet18_v3_yolov8mix.pth"
best_val_f1 = -1

import time
for epoch in range(num_epochs):
    t0 = time.time()
    tr_loss, tr_acc = train_one_epoch(model_v3, train_loader, criterion, optimizer, scaler, device)
    vp, vl = collect_probs_and_labels(model_v3, val_loader, device, pothole_idx)
    p, r, f1, _ = metrics_from_threshold(vl, vp, selection_threshold, pothole_idx)
    scheduler.step(f1)
    dt = time.time() - t0
    print(f"Epoch [{epoch+1}/{num_epochs}] {dt:.1f}s | loss={tr_loss:.4f} acc={tr_acc:.4f} | "
          f"val P={p:.4f} R={r:.4f} F1={f1:.4f}")
    if f1 > best_val_f1:
        best_val_f1 = f1
        torch.save(model_v3.state_dict(), save_path_v3)
        print("  saved ->", save_path_v3)

C:\Users\pc\AppData\Local\Temp\ipykernel_5316\2128071325.py:92: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = GradScaler()
C:\Users\pc\AppData\Local\Temp\ipykernel_5316\2128071325.py:42: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=torch.float16):
C:\Users\pc\AppData\Local\Temp\ipykernel_5316\2128071325.py:59: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=torch.float16):


Epoch [1/10] 257.6s | loss=0.2902 acc=0.8812 | val P=0.8319 R=0.7799 F1=0.8050
  saved -> best_resnet18_v3_yolov8mix.pth


C:\Users\pc\AppData\Local\Temp\ipykernel_5316\2128071325.py:42: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=torch.float16):
C:\Users\pc\AppData\Local\Temp\ipykernel_5316\2128071325.py:59: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=torch.float16):


Epoch [2/10] 138.3s | loss=0.1947 acc=0.9273 | val P=0.8450 R=0.7853 F1=0.8141
  saved -> best_resnet18_v3_yolov8mix.pth


C:\Users\pc\AppData\Local\Temp\ipykernel_5316\2128071325.py:42: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=torch.float16):
C:\Users\pc\AppData\Local\Temp\ipykernel_5316\2128071325.py:59: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=torch.float16):


...

Epoch [8/10] 135.7s | loss=0.0390 acc=0.9869 | val P=0.9078 R=0.7228 F1=0.8048


C:\Users\pc\AppData\Local\Temp\ipykernel_5316\2128071325.py:42: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=torch.float16):
C:\Users\pc\AppData\Local\Temp\ipykernel_5316\2128071325.py:59: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=torch.float16):


Epoch [9/10] 138.7s | loss=0.0319 acc=0.9890 | val P=0.9000 R=0.7826 F1=0.8372
  saved -> best_resnet18_v3_yolov8mix.pth


C:\Users\pc\AppData\Local\Temp\ipykernel_5316\2128071325.py:42: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=torch.float16):
C:\Users\pc\AppData\Local\Temp\ipykernel_5316\2128071325.py:59: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=torch.float16):


Epoch [10/10] 137.8s | loss=0.0288 acc=0.9895 | val P=0.8925 R=0.7446 F1=0.8119


In [5]:
import numpy as np
import torch
from torch.utils.data import DataLoader
from torch.amp import autocast
from sklearn.metrics import precision_score, recall_score, f1_score

# 训练时可以多 worker；评估时在 Windows/Jupyter 里最稳的是 0
bs_eval = 128

val_loader = DataLoader(
    val_dataset, batch_size=bs_eval, shuffle=False,
    num_workers=0, pin_memory=True
)
test_loader = DataLoader(
    test_dataset, batch_size=bs_eval, shuffle=False,
    num_workers=0, pin_memory=True
)
challenge_loader = DataLoader(
    challenge_dataset, batch_size=bs_eval, shuffle=False,
    num_workers=0, pin_memory=True
)
external_sovit_neg_loader = DataLoader(
    external_sovit_neg_dataset, batch_size=bs_eval, shuffle=False,
    num_workers=0, pin_memory=True
)
yolov8_holdout_loader = DataLoader(
    yolov8_holdout_dataset, batch_size=bs_eval, shuffle=False,
    num_workers=0, pin_memory=True
)

@torch.no_grad()
def collect_probs_and_labels(model, loader, device, pothole_idx):
    model.eval()
    ps, ls = [], []
    for images, labels in loader:
        images = images.to(device, non_blocking=True, memory_format=torch.channels_last)
        with autocast(device_type="cuda", dtype=torch.float16):
            out = model(images)
        probs = torch.softmax(out.float(), dim=1)[:, pothole_idx]
        ps.extend(probs.cpu().numpy().tolist())
        ls.extend(labels.numpy().tolist())
    return np.array(ps), np.array(ls)

@torch.no_grad()
def collect_probs_only(model, loader, device, pothole_idx):
    model.eval()
    ps = []
    for images, _ in loader:
        images = images.to(device, non_blocking=True, memory_format=torch.channels_last)
        with autocast(device_type="cuda", dtype=torch.float16):
            out = model(images)
        probs = torch.softmax(out.float(), dim=1)[:, pothole_idx]
        ps.extend(probs.cpu().numpy().tolist())
    return np.array(ps)

def metrics_from_threshold(y_true, y_prob, threshold, pothole_idx):
    y_pred = (y_prob >= threshold).astype(int)
    return (
        precision_score(y_true, y_pred, pos_label=pothole_idx),
        recall_score(y_true, y_pred, pos_label=pothole_idx),
        f1_score(y_true, y_pred, pos_label=pothole_idx),
        y_pred,
    )

print("Eval loaders rebuilt with num_workers=0")

Eval loaders rebuilt with num_workers=0


In [6]:
import pandas as pd

model_v3.load_state_dict(torch.load(save_path_v3, map_location=device))
model_v3.eval()

model_old = models.resnet18(weights=None)
model_old.fc = nn.Linear(model_old.fc.in_features, 2)
model_old.load_state_dict(torch.load("best_resnet18_v2.pth", map_location=device))
model_old = model_old.to(device).eval()

def eval_all(model):
    tp, tl = collect_probs_and_labels(model, test_loader,              device, pothole_idx)
    ep     = collect_probs_only     (model, challenge_loader,          device, pothole_idx)
    sp     = collect_probs_only     (model, external_sovit_neg_loader, device, pothole_idx)
    yp     = collect_probs_only     (model, yolov8_holdout_loader,     device, pothole_idx)
    return tp, tl, ep, sp, yp

old = eval_all(model_old)
new = eval_all(model_v3)

thresholds = [0.45, 0.50, 0.55, 0.60]

def summarize(name, bundle):
    tp, tl, ep, sp, yp = bundle
    out = []
    for th in thresholds:
        p, r, f1, _ = metrics_from_threshold(tl, tp, th, pothole_idx)
        out.append({
            "model": name, "threshold": th,
            "rdd_test_precision": round(p, 4),
            "rdd_test_recall":    round(r, 4),
            "rdd_test_f1":        round(f1, 4),
            "excl_alligator_fp":  round(float(np.mean(ep >= th)), 4),
            "sovit_neg_fp":       round(float(np.mean(sp >= th)), 4),
            "yolov8_holdout_recall":    round(float(np.mean(yp >= th)), 4),
            "yolov8_holdout_miss_rate": round(1 - float(np.mean(yp >= th)), 4),
        })
    return out

rows = []
rows += summarize("v2 (no YOLOv8 in train)", old)
rows += summarize("v3 (+ YOLOv8 in train)",  new)
df_v3 = pd.DataFrame(rows)
print(df_v3.to_string(index=False))

                  model  threshold  rdd_test_precision  rdd_test_recall  rdd_test_f1  excl_alligator_fp  sovit_neg_fp  yolov8_holdout_recall  yolov8_holdout_miss_rate
v2 (no YOLOv8 in train)       0.45              0.8937           0.8474       0.8699             0.3359        0.0132                 0.6923                    0.3077
v2 (no YOLOv8 in train)       0.50              0.8934           0.8447       0.8683             0.3281        0.0132                 0.6624                    0.3376
v2 (no YOLOv8 in train)       0.55              0.9053           0.8338       0.8681             0.3182        0.0113                 0.6496                    0.3504
v2 (no YOLOv8 in train)       0.60              0.9066           0.8202       0.8612             0.3064        0.0113                 0.6368                    0.3632
 v3 (+ YOLOv8 in train)       0.45              0.9147           0.8474       0.8798             0.3011        0.0188                 0.9957                    0.004

In [7]:
from pathlib import Path
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader

neha_root = Path(r"D:\Big_Data\Neha Pothole and normal dataset")
image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

print("root exists:", neha_root.exists())
print("root is dir:", neha_root.is_dir())

pothole_folders = []
normal_folders  = []
for p in neha_root.rglob("*"):
    if not p.is_dir():
        continue
    has_imgs = any(q.is_file() and q.suffix.lower() in image_exts for q in p.iterdir())
    if not has_imgs:
        continue
    name_low = p.name.lower()
    if "pothole" in name_low:
        pothole_folders.append(p)
    elif "normal" in name_low or "plain" in name_low or "non" in name_low:
        normal_folders.append(p)

print("\nDetected pothole folders:")
for f in pothole_folders: print("  ", f)
print("Detected normal folders:")
for f in normal_folders: print("  ", f)

pothole_files = []
for f in pothole_folders:
    pothole_files += [p for p in f.iterdir() if p.is_file() and p.suffix.lower() in image_exts]
normal_files = []
for f in normal_folders:
    normal_files += [p for p in f.iterdir() if p.is_file() and p.suffix.lower() in image_exts]

print(f"\npothole images: {len(pothole_files)}")
print(f"normal  images: {len(normal_files)}")

assert len(pothole_files) > 0 and len(normal_files) > 0, \
    "Didn't detect both classes. 请把上面的 detected folders 发我，我改匹配规则。"

class BinaryPotholeDataset(Dataset):
    def __init__(self, pothole_paths, normal_paths, transform, pothole_label):
        self.items = [(p, pothole_label) for p in pothole_paths] \
                   + [(p, 1 - pothole_label) for p in normal_paths]
        self.transform = transform
    def __len__(self):  return len(self.items)
    def __getitem__(self, idx):
        p, y = self.items[idx]
        img = Image.open(p).convert("RGB")
        return self.transform(img), y

neha_dataset = BinaryPotholeDataset(
    pothole_files, normal_files,
    transform=eval_transform,
    pothole_label=pothole_idx,
)
neha_loader = DataLoader(
    neha_dataset, batch_size=128, shuffle=False,
    num_workers=0, pin_memory=True,
)
print("Neha eval dataset size:", len(neha_dataset))

root exists: True
root is dir: True

Detected pothole folders:
   D:\Big_Data\Neha Pothole and normal dataset\Pothole
Detected normal folders:
   D:\Big_Data\Neha Pothole and normal dataset\Normal

pothole images: 2500
normal  images: 2500
Neha eval dataset size: 5000


In [8]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix

# 拿到所有样本的 pothole 概率和真实 label（0=normal, 1=pothole 按 pothole_idx 对齐）
v2_probs, neha_labels = collect_probs_and_labels(model_old, neha_loader, device, pothole_idx)
v3_probs, _           = collect_probs_and_labels(model_v3,  neha_loader, device, pothole_idx)

n_pos = int(np.sum(neha_labels == pothole_idx))
n_neg = int(np.sum(neha_labels != pothole_idx))
print(f"Neha dataset — pothole (positive): {n_pos}, normal (negative): {n_neg}")

thresholds = [0.45, 0.50, 0.55, 0.60]

def eval_one(name, probs, labels):
    out = []
    for th in thresholds:
        p, r, f1, y_pred = metrics_from_threshold(labels, probs, th, pothole_idx)
        is_pos = (labels == pothole_idx)
        is_neg = ~is_pos
        pred_pos = (y_pred == pothole_idx)

        # FN rate = positives missed
        fn_rate = 1.0 - r
        # FP rate = negatives flagged as pothole
        fp_rate = float(np.mean(pred_pos[is_neg]))
        # overall accuracy
        acc = float(np.mean(pred_pos == is_pos))

        out.append({
            "model": name,
            "threshold": th,
            "accuracy": round(acc, 4),
            "precision": round(p, 4),
            "recall (pothole)": round(r, 4),
            "f1 (pothole)": round(f1, 4),
            "miss_rate (FN)": round(fn_rate, 4),
            "fp_rate (FP)": round(fp_rate, 4),
        })
    return out

rows = []
rows += eval_one("v2 (no YOLOv8)",    v2_probs, neha_labels)
rows += eval_one("v3 (+ YOLOv8)",     v3_probs, neha_labels)

df_neha = pd.DataFrame(rows)
print("\n=== External test on Neha Normal/Pothole dataset ===")
print(df_neha.to_string(index=False))

print("\n--- Confusion matrices at threshold 0.50 (rows=true [neg,pos], cols=pred [neg,pos]) ---")
for name, probs in [("v2", v2_probs), ("v3", v3_probs)]:
    y_pred_50 = (probs >= 0.50).astype(int)
    y_true    = (neha_labels == pothole_idx).astype(int)
    cm = confusion_matrix(y_true, y_pred_50, labels=[0, 1])
    print(f"{name}:\n{cm}")

Neha dataset — pothole (positive): 2500, normal (negative): 2500

=== External test on Neha Normal/Pothole dataset ===
         model  threshold  accuracy  precision  recall (pothole)  f1 (pothole)  miss_rate (FN)  fp_rate (FP)
v2 (no YOLOv8)       0.45    0.8108     0.9255            0.6760        0.7813          0.3240        0.0544
v2 (no YOLOv8)       0.50    0.8062     0.9293            0.6628        0.7738          0.3372        0.0504
v2 (no YOLOv8)       0.55    0.8026     0.9345            0.6508        0.7673          0.3492        0.0456
v2 (no YOLOv8)       0.60    0.7998     0.9386            0.6416        0.7622          0.3584        0.0420
 v3 (+ YOLOv8)       0.45    0.8800     0.8569            0.9124        0.8838          0.0876        0.1524
 v3 (+ YOLOv8)       0.50    0.8824     0.8649            0.9064        0.8852          0.0936        0.1416
 v3 (+ YOLOv8)       0.55    0.8854     0.8739            0.9008        0.8871          0.0992        0.1300
 v3 (+ YO

In [9]:
import shutil, numpy as np, pandas as pd
from pathlib import Path

out_dir = Path(r"D:\Big_Data\Neha_v3_top_FPs")
out_dir.mkdir(parents=True, exist_ok=True)

threshold = 0.50
is_true_negative = (neha_labels != pothole_idx)
pred_pos = (v3_probs >= threshold)
fp_mask = is_true_negative & pred_pos

paths = [p for p, _ in neha_dataset.items]
rows = [{"path": str(paths[i]), "prob": float(v3_probs[i])}
        for i in np.where(fp_mask)[0]]
df_fp = pd.DataFrame(rows).sort_values("prob", ascending=False).reset_index(drop=True)

top_k = min(50, len(df_fp))
for _, r in df_fp.head(top_k).iterrows():
    src = Path(r["path"])
    shutil.copy2(src, out_dir / f"{r['prob']:.3f}_{src.name}")

print(f"v3 FP on Neha normals: {len(df_fp)}")
print(f"top {top_k} copied to: {out_dir}")

v3 FP on Neha normals: 354
top 50 copied to: D:\Big_Data\Neha_v3_top_FPs


In [10]:
import json
from pathlib import Path
import torch

export_dir = Path(r"D:\Big_Data\pothole_model_v3_export")
export_dir.mkdir(parents=True, exist_ok=True)

torch.save(model_v3.state_dict(), export_dir / "weights.pth")

config = {
    "arch": "resnet18",
    "num_classes": 2,
    "class_to_idx": train_dataset.class_to_idx,
    "pothole_idx": int(pothole_idx),
    "default_threshold": 0.50,
    "input_size": 224,
    "normalize_mean": [0.485, 0.456, 0.406],
    "normalize_std":  [0.229, 0.224, 0.225],
    "training_sources": [
        "RDD2020 (relabelled D40 pothole / D00+D10 as non_pothole)",
        "Sovit Simplex pothole dataset",
        "Pothole_Segmentation_YOLOv8 (70% of positive images)",
    ],
    "external_test_results": {
        "neha_normal_pothole_dataset @ threshold=0.50": {
            "accuracy": 0.8824, "recall_pothole": 0.9064,
            "precision": 0.8649, "f1": 0.8852,
            "fn_rate": 0.0936, "fp_rate": 0.1416
        }
    },
    "notes": "Binary pothole classifier. Input: RGB road image. Output: pothole probability."
}

with open(export_dir / "config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print("Exported to:", export_dir)
for p in sorted(export_dir.iterdir()):
    print(" ", p.name, f"({p.stat().st_size/1e6:.2f} MB)")

Exported to: D:\Big_Data\pothole_model_v3_export
  config.json (0.00 MB)
  weights.pth (44.79 MB)


In [11]:
# pothole_classifier.py
import json
from pathlib import Path
from typing import Union, List, Sequence
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image


class PotholeClassifier:
    """
    Usage:
        clf = PotholeClassifier(r"path/to/pothole_model_v3_export")
        r = clf.predict("some_image.jpg")
        # -> {"pothole_probability": 0.87, "is_pothole": True, "threshold": 0.5}
    """

    def __init__(self, export_dir: Union[str, Path],
                 device: Union[str, torch.device, None] = None,
                 threshold: Union[float, None] = None):
        export_dir = Path(export_dir)
        with open(export_dir / "config.json", "r", encoding="utf-8") as f:
            self.config = json.load(f)

        self.device = torch.device(device) if device else \
                      torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.pothole_idx = int(self.config["pothole_idx"])
        self.threshold = float(
            threshold if threshold is not None else self.config["default_threshold"]
        )

        model = models.resnet18(weights=None)
        model.fc = nn.Linear(model.fc.in_features, self.config["num_classes"])
        sd = torch.load(export_dir / "weights.pth", map_location=self.device)
        model.load_state_dict(sd)
        self.model = model.to(self.device).eval()

        size = int(self.config["input_size"])
        self.transform = transforms.Compose([
            transforms.Resize((size, size)),
            transforms.ToTensor(),
            transforms.Normalize(self.config["normalize_mean"],
                                 self.config["normalize_std"]),
        ])

    @torch.no_grad()
    def predict(self, image):
        single = not isinstance(image, (list, tuple))
        imgs = [image] if single else list(image)

        tensors = []
        for im in imgs:
            if isinstance(im, (str, Path)):
                im = Image.open(im).convert("RGB")
            elif isinstance(im, Image.Image):
                im = im.convert("RGB")
            else:
                raise TypeError(f"Unsupported input type: {type(im)}")
            tensors.append(self.transform(im))

        batch = torch.stack(tensors).to(self.device)
        probs = torch.softmax(self.model(batch), dim=1)[:, self.pothole_idx]
        probs = probs.cpu().numpy().tolist()

        results = [{
            "pothole_probability": float(p),
            "is_pothole": bool(p >= self.threshold),
            "threshold": self.threshold,
        } for p in probs]
        return results[0] if single else results

In [12]:
import torch
from pathlib import Path

export_dir = Path(r"D:\Big_Data\pothole_model_v3_export")
export_dir.mkdir(parents=True, exist_ok=True)

# make sure model is in eval mode
model_v3.eval()

# create one dummy input with the same size as inference input
example = torch.randn(1, 3, 224, 224, device=device)

# trace and save
with torch.no_grad():
    traced_model = torch.jit.trace(model_v3, example)
    traced_model.save(str(export_dir / "model_v3_traced.pt"))

print("Saved traced model to:")
print(export_dir / "model_v3_traced.pt")

Saved traced model to:
D:\Big_Data\pothole_model_v3_export\model_v3_traced.pt
